In [51]:
import cv2
import mediapipe as mp
import time
from tqdm import tqdm
import numpy as np

In [52]:
# 导入solution
mp_pose = mp.solutions.pose

# 导入绘图函数
mp_drawing = mp.solutions.drawing_utils

# 导入模型
pose = mp_pose.Pose(
    static_image_mode=False,  # 静态图片 or 连续帧
    model_complexity=2,  # 模型复杂度 0最快 2最好
    smooth_landmarks=True,  # 是否平滑关键点
    min_detection_confidence=0.5,  # 置信度阈值
    min_tracking_confidence=0.8  # 追踪阈值
)

In [53]:
# 处理帧函数
# 输入摄像头实时画面  返回推理并可视化的图片
def process_frame(img):
    start_time = time.time()

    h, w = img.shape[0], img.shape[1]

    img_RGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = pose.process(img_RGB)
    # print(results.pose_landmarks)
    # save_results(results.pose_landmarks)
     
    if results.pose_landmarks:
        # connection_drawing_spec = mp_drawing.DrawingSpec(color=(225, 225, 225), thickness=3)
        mp_drawing.draw_landmarks(img, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        # mp_drawing.draw_landmarks(img, results.pose_landmarks,
        #                           mp_pose.POSE_CONNECTIONS,
        #                           connection_drawing_spec=connection_drawing_spec)
        # 
        # for i in range(33):
        #     x = int(results.pose_landmarks.landmark[i].x * w)
        #     y = int(results.pose_landmarks.landmark[i].y * h)
        #     z = results.pose_landmarks.landmark[i].z
        # 
        #     radius = 10
        # 
        #     if i == 0:  # 鼻尖
        #         img = cv2.circle(img, (x, y), 5, (0, 0, 255), -1)
        #     elif i in [11, 12]:  # 肩膀
        #         img = cv2.circle(img, (x, y), radius, (223, 155, 6), -1)
        #     elif i in [23, 24]:  # 髋关节
        #         img = cv2.circle(img, (x, y), radius, (0, 240, 255), -1)
        #     elif i in [13, 14]:  # 胳膊肘
        #         img = cv2.circle(img, (x, y), radius, (140, 47, 240), -1)
        #     elif i in [25, 26]:  # 膝盖
        #         img = cv2.circle(img, (x, y), radius, (0, 0, 255), -1)
        #     elif i in [15, 16, 27, 28]:  # 手腕和脚腕
        #         img = cv2.circle(img, (x, y), radius, (223, 155, 60), -1)
        #     elif i in [17, 19, 21]:  # 左手
        #         img = cv2.circle(img, (x, y), radius, (94, 218, 121), -1)
        #     elif i in [18, 20, 22]:  # 右手
        #         img = cv2.circle(img, (x, y), radius, (16, 144, 247), -1)
        #     elif i in [27, 29, 31]:  # 左脚
        #         img = cv2.circle(img, (x, y), radius, (29, 123, 243), -1)
        #     elif i in [28, 30, 32]:  # 右脚
        #         img = cv2.circle(img, (x, y), radius, (193, 182, 255), -1)
        #     elif i in [9, 10]:  # 嘴
        #         img = cv2.circle(img, (x, y), 5, (205, 235, 255), -1)
        #     elif i in [1, 2, 3, 4, 5, 6, 7, 8]:  # 眼部
        #         img = cv2.circle(img, (x, y), 5, (94, 218, 121), -1)
        #     else:  # 其他关键点
        #         img = cv2.circle(img, (x, y), 5, (0, 255, 0), -1)

    end_time = time.time()
    FPS = 1 / (end_time - start_time)

    scaler = 1

    img = cv2.putText(img, 'FPS  ' + str(int(FPS)), (25 * scaler, 50 * scaler), cv2.FONT_HERSHEY_SIMPLEX, 1.25 * scaler,
                      (0, 0, 255), 2)

    return img

In [54]:
# 视频帧处理
def generate_video(input_path):
    filehead = input_path.split('/')[-1]
    output_path = "out-" + filehead

    print("视频开始处理", input_path)

    # 获取视频总帧数
    cap = cv2.VideoCapture(input_path)
    frame_count = 0
    while cap.isOpened():
        success, frame = cap.read()
        frame_count += 1
        if not success:
            break
        cap.release()
        # print("视频总帧数为", frame_count)

        cap = cv2.VideoCapture(input_path)
        frame_size = (cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        fourcc = cv2.VideoWriter.fourcc(*'mp4v')
        fps = cap.get(cv2.CAP_PROP_FPS)

        out = cv2.VideoWriter(output_path, fourcc, fps, (int(frame_size[0]), int(frame_size[1])))

        # 进度条绑定视频总帧数
        with tqdm(total=frame_count - 1) as pbar:
            try:
                while cap.isOpened():
                    success, frame = cap.read()
                    if not success:
                        break

                    # 处理帧
                    try:
                        frame = process_frame(frame)
                    except:
                        print("error")
                        pass

                    if success:
                        out.write(frame)

                        pbar.update(1)
            except:
                print("中途中断")
                pass

        cv2.destroyAllWindows()
        out.release()
        cap.release()
        print("视频已保存", output_path)


In [55]:
video_path = "data/416.mp4"
generate_video(video_path)

视频开始处理 data/416.mp4


306it [01:03,  4.82it/s]

视频已保存 out-416.mp4
